In [1]:
import numpy as np
from pathlib import Path

In [2]:
# ==========================================================
# IPF / Furness Algorithm
# ==========================================================
def ipf(seed_matrix,
        target_row,
        target_col,
        tol=1e-3,
        max_iter=1000):

    T = seed_matrix.astype(float).copy()

    for _ in range(max_iter):

        # Row balancing
        row_sum = T.sum(axis=1)
        row_factor = np.divide(
            target_row,
            row_sum,
            out=np.ones_like(target_row),
            where=row_sum > 0
        )
        T *= row_factor[:, None]

        # Column balancing
        col_sum = T.sum(axis=0)
        col_factor = np.divide(
            target_col,
            col_sum,
            out=np.ones_like(target_col),
            where=col_sum > 0
        )
        T *= col_factor[None, :]

        row_err = np.max(np.abs(T.sum(axis=1) - target_row))
        col_err = np.max(np.abs(T.sum(axis=0) - target_col))

        if max(row_err, col_err) < tol:
            break

    return T

In [3]:
# ==========================================================
# Generate Perturbed Marginals
# ==========================================================
def perturb_marginals(
        productions,
        attractions,
        perturbation=0.20,
        rng=None):

    if rng is None:
        rng = np.random.default_rng()

    # perturb productions
    p_new = productions * (
        1 + rng.uniform(
            -perturbation,
            perturbation,
            size=productions.shape
        )
    )

    # perturb attractions
    a_new = attractions * (
        1 + rng.uniform(
            -perturbation,
            perturbation,
            size=attractions.shape
        )
    )

    # enforce same grand total
    total = productions.sum()

    p_new *= total / p_new.sum()
    a_new *= total / a_new.sum()

    return p_new, a_new

In [4]:
# ==========================================================
# Generate One Synthetic OD Matrix
# ==========================================================
def generate_synthetic_od(
        base_od,
        perturbation=0.20,
        seed=None):

    rng = np.random.default_rng(seed)

    N = base_od.shape[0]

    productions = base_od.sum(axis=1)
    attractions = base_od.sum(axis=0)

    # Preserve sparsity structure
    mask = (base_od > 0).astype(float)

    # Perturb marginals
    target_row, target_col = perturb_marginals(
        productions,
        attractions,
        perturbation,
        rng
    )

    # Seed matrix
    seed_matrix = base_od.copy().astype(float)

    # Small multiplicative perturbation
    noise = rng.uniform(
        0.8,
        1.2,
        size=seed_matrix.shape
    )

    seed_matrix *= noise

    # Preserve sparsity
    seed_matrix *= mask

    # No intrazonal trips
    np.fill_diagonal(seed_matrix, 0)

    # Tiny value avoids divide-by-zero in IPF
    seed_matrix += mask * 1e-6

    # Run IPF
    synthetic = ipf(
        seed_matrix,
        target_row,
        target_col
    )

    # Restore exact sparsity
    synthetic *= mask

    # Force diagonal zero again
    np.fill_diagonal(synthetic, 0)

    return synthetic

In [6]:
# ==========================================================
# Main
# ==========================================================
if __name__ == "__main__":

    base_od = np.load("../data/EstimatedODMatrix.npy").astype(float)

    # Ensure non-negative
    base_od = np.clip(base_od, 0, None)

    # Force diagonal zero
    np.fill_diagonal(base_od, 0)

    synthetic_matrices = []

    for i in range(100):

        od = generate_synthetic_od(
            base_od,
            perturbation=0.20,
            seed=1000 + i
        )

        synthetic_matrices.append(od)

    synthetic_matrices = np.array(synthetic_matrices)

    print(
        f"Generated {synthetic_matrices.shape[0]} matrices "
        f"of size {synthetic_matrices.shape[1]}x"
        f"{synthetic_matrices.shape[2]}"
    )

    np.save(
        "synthetic_od_100.npy",
        synthetic_matrices
    )

    print(
        "Saved synthetic_od_100.npy"
    )

Generated 100 matrices of size 24x24
Saved synthetic_od_100.npy


In [10]:
synthetic_matrices[0]

array([[0.00000000e+00, 8.90040446e+01, 9.91765659e+01, 5.84078510e+02,
        3.08316560e+02, 4.55129238e+02, 4.19298819e+02, 6.90395302e+02,
        5.35339991e+02, 1.32291881e+03, 3.77297532e+02, 2.07438606e+02,
        4.74286396e+02, 3.61838799e+02, 3.83022893e+02, 5.49519338e+02,
        3.60836599e+02, 8.81457854e+01, 2.49614034e+02, 2.36090988e+02,
        7.48563785e+01, 3.81136969e+02, 3.19491856e+02, 1.18286495e+02],
       [7.16709462e+01, 0.00000000e+00, 7.63044601e+01, 2.41789793e+02,
        1.92300533e+02, 4.16834270e+02, 1.90003244e+02, 4.04992195e+02,
        2.16510968e+02, 6.02447455e+02, 1.94521208e+02, 8.39993730e+01,
        3.52481827e+02, 8.50336673e+01, 6.73929765e+01, 4.19069980e+02,
        2.13536708e+02, 1.56218694e+01, 1.07949971e+02, 6.52735614e+01,
        1.38898722e+01, 1.08642943e+02, 1.36469189e+00, 3.96618492e-01],
       [8.33881297e+01, 1.06917940e+02, 0.00000000e+00, 1.92975293e+02,
        1.60652727e+02, 3.55398202e+02, 5.05124574e+01, 2.0047

In [11]:
base_od

array([[0.00000000e+00, 9.99999887e+01, 9.99999942e+01, 5.39854233e+02,
        2.14176996e+02, 3.62379376e+02, 4.92118093e+02, 8.19343637e+02,
        4.94916023e+02, 1.28377146e+03, 5.14455086e+02, 2.41342610e+02,
        5.40896020e+02, 2.92144342e+02, 4.60681924e+02, 4.84786752e+02,
        3.90970948e+02, 9.27743741e+01, 2.69255341e+02, 2.88371434e+02,
        8.17624441e+01, 3.48389591e+02, 2.76605758e+02, 1.08082561e+02],
       [9.99999927e+01, 0.00000000e+00, 9.70792359e+01, 2.25960882e+02,
        1.37414596e+02, 3.99999994e+02, 2.24738717e+02, 4.73964261e+02,
        1.70707385e+02, 5.75123876e+02, 1.97613883e+02, 9.74218456e+01,
        3.03975256e+02, 6.83031390e+01, 7.45196358e+01, 4.29407376e+02,
        1.93591572e+02, 1.93949980e+01, 1.04875965e+02, 6.99920582e+01,
        1.62006414e+01, 8.05703488e+01, 1.14799752e+00, 3.75775594e-01],
       [9.99999915e+01, 1.25707247e+02, 0.00000000e+00, 1.99999991e+02,
        1.58322763e+02, 3.19046276e+02, 6.73622720e+01, 2.2058